<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Solutions</h2>
<h2>Notebook C03: Ensembles</h2>
</div>

Worked solutions to the 2 exercises in
[Notebook C03: Ensembles](../notebooks/C03_Ensembles.ipynb).

**Try each exercise yourself first.** These notebooks are most useful as a check on your reasoning, and
least useful as something to read straight through. An exercise you attempted and got wrong teaches more
than a solution you agreed with.

Where an exercise asks a question rather than requesting code, the answer is written out under the code
that produces it. Several of them have answers that are more interesting than they look.

The setup cell below reproduces the state the exercises assume, so this notebook runs on its own.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="setup">Setup</h3>
</div>

The four base models, the three-way split and the weight optimiser from the notebook. The setup cell fits
every model once and takes a minute or so, mostly SARIMAX.

In [ ]:
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from scipy.optimize import minimize
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.arima.model import ARIMA

sys.path.append("../notebooks")
import nb_config

sns.set_theme(style="whitegrid")

sales = pd.read_csv(nb_config.ROSSMANN_TRAIN_PATH, parse_dates=["Date"], low_memory=False)
store = sales[sales["Store"] == 1].set_index("Date").sort_index().asfreq("D")
target = store["Sales"].astype(float)


def build_features(target, store):
    features = pd.DataFrame(index=target.index)

    for lag in (1, 2, 7, 14, 28):
        features[f"lag_{lag}"] = target.shift(lag)

    history = target.shift(1)
    for window in (7, 28):
        features[f"roll_mean_{window}"] = history.rolling(window).mean()
        features[f"roll_std_{window}"] = history.rolling(window).std()

    features["day_of_week"] = target.index.dayofweek
    features["day_of_month"] = target.index.day
    features["month"] = target.index.month
    features["days_since_start"] = (target.index - target.index[0]).days

    for k in (1, 2):
        position = target.index.dayofyear / 365.25
        features[f"fourier_sin_{k}"] = np.sin(2 * np.pi * k * position)
        features[f"fourier_cos_{k}"] = np.cos(2 * np.pi * k * position)

    features["open"] = store["Open"]
    features["promo"] = store["Promo"]
    features["school_holiday"] = store["SchoolHoliday"]

    return features


features = build_features(target, store)
complete = features.notna().all(axis=1)
X, y = features[complete], target[complete]
exogenous = store[["Open", "Promo"]].astype(float).loc[X.index]

VALIDATION_DAYS = 90
TEST_DAYS = 90
test_start = len(X) - TEST_DAYS
validation_start = test_start - VALIDATION_DAYS


def fit_base_models(X, y, exogenous, validation_start, test_start, horizon):
    """Four forecasters, with their validation and test predictions."""
    learners = {
        "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=10.0)),
        "Random forest": RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1),
        "LightGBM": lgb.LGBMRegressor(
            n_estimators=200, learning_rate=0.03, num_leaves=7, random_state=0, verbose=-1
        ),
    }
    validation, test = {}, {}

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        for name, learner in learners.items():
            learner.fit(X.iloc[:validation_start], y.iloc[:validation_start])
            validation[name] = learner.predict(X.iloc[validation_start:test_start])
            test[name] = learner.predict(X.iloc[test_start:test_start + horizon])

        for start, end, store_in in [(validation_start, test_start, validation),
                                     (test_start, test_start + horizon, test)]:
            model = ARIMA(
                y.iloc[:start], exog=exogenous.iloc[:start],
                order=(1, 0, 1), seasonal_order=(1, 0, 1, 7),
            ).fit()
            store_in["SARIMAX"] = model.forecast(end - start, exog=exogenous.iloc[start:end]).values

    return (pd.DataFrame(validation, index=X.index[validation_start:test_start]),
            pd.DataFrame(test, index=X.index[test_start:test_start + horizon]))


def optimal_weights(errors):
    """Non-negative weights summing to one that minimise the mean absolute error."""
    n_models = errors.shape[1]
    result = minimize(
        lambda weights: np.mean(np.abs(errors @ weights)),
        x0=np.ones(n_models) / n_models,
        bounds=[(0.0, 1.0)] * n_models,
        constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1.0}],
    )
    return pd.Series(result.x, index=errors.columns)


validation_predictions, test_predictions = fit_base_models(
    X, y, exogenous, validation_start, test_start, TEST_DAYS
)
y_validation = y.iloc[validation_start:test_start]
y_test = y.iloc[test_start:]

validation_errors = validation_predictions.sub(y_validation, axis=0)
individual = pd.Series(
    {name: mean_absolute_error(y_test, test_predictions[name]) for name in test_predictions}
).sort_values()

print(individual.round(1).to_string())

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-1">Exercise 1</h3>
</div>

> Refit the weights with `Ridge` and `SARIMAX` removed from the pool entirely. Do the weights on the two tree models change, and does the test score move? What does that tell you about how much the zero-weighted models were contributing?

In [ ]:
TREES = ["Random forest", "LightGBM"]

full_pool = optimal_weights(validation_errors)
trees_only = optimal_weights(validation_errors[TREES])

weights = pd.DataFrame({"Four models": full_pool, "Two trees": trees_only})

scores = pd.Series({
    "Four models": mean_absolute_error(y_test, test_predictions.values @ full_pool.values),
    "Two trees": mean_absolute_error(y_test, test_predictions[TREES].values @ trees_only.values),
})

print(weights.round(4).to_string())
print()
print(scores.round(4).to_string())
print(f"\ndifference: {abs(scores.iloc[0] - scores.iloc[1]):.2e} MAE")

**Nothing changes. The weights are identical to four decimal places, and so is the test score: 231.1
either way.**

That is not a coincidence or an approximation. Look at the four-model weights the notebook already
produced: Ridge 0.000, SARIMAX 0.000. The constrained optimiser had **already removed them**. Dropping a
model whose weight is exactly zero from a weighted average cannot change the average, so the remaining
weights are solving an identical problem and land in an identical place.

So the direct answer to "how much were the zero-weighted models contributing?" is **exactly nothing, and
you can verify it to the optimiser's tolerance rather than argue about it.**

Which raises the obvious question: why were they in the pool? Their validation errors give the answer.

In [ ]:
diagnostics = pd.DataFrame({
    "Validation MAE": validation_errors.abs().mean(),
    "Test MAE": individual,
    "Weight": full_pool,
})

print(diagnostics.round(1).to_string())
print()
print("Error correlation on the validation period:")
print(validation_errors.corr().round(2).to_string())

Ridge and SARIMAX are not marginally worse than the trees — they are **roughly 75% worse**, at 551 and
568 validation MAE against 310 and 330. A weighted average can only help if the weak model's errors offset
the strong one's, and theirs do not: SARIMAX correlates with Ridge at 0.89 and with LightGBM at 0.83. They
are wrong at the same times, in the same direction, only by more. There is nothing to offset.

Two things follow, and the second is more useful than the first.

**The optimiser did the selection for you.** With non-negative weights constrained to sum to one, a model
that cannot improve the objective receives zero, and zero is a genuine exclusion rather than a small
number. That is exactly why the notebook constrained the weights. An unconstrained least-squares fit would
have handed Ridge and SARIMAX some nonzero coefficient — probably a negative one — and quietly made the
combination depend on two models it should ignore.

**But "no effect on the score" is not "no effect".** Dropping them removes two models from the pipeline,
and SARIMAX is the slowest thing in this notebook by a wide margin: it is refitted from scratch at every
origin while the tree models are fitted once. Removing it costs nothing in accuracy and saves most of the
runtime. In production the same reasoning applies to everything downstream — two fewer models to monitor,
retrain, and explain when they break.

One caution before you delete them for good. **Zero weight on one validation window is not zero weight
forever.** The weights are fitted on 90 days; a different 90 days can produce a different answer, and a
model that is useless in a stable period is sometimes the one that holds up when the regime changes. The
measured claim is "these two contributed nothing here", not "these two are useless". Exercise 2 is a first
test of how much that distinction matters.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-2">Exercise 2</h3>
</div>

> Drop the random forest from the pool and rerun the comparison across origins. Does the ensemble now beat the best remaining individual model? What does the answer suggest about when combination is most valuable?

In [ ]:
HORIZON = 60
N_ORIGINS = 6


def score_pool(validation_predictions, test_predictions, y_validation, y_test, pool):
    """Every individual model and every combination, restricted to `pool`."""
    validation_predictions = validation_predictions[pool]
    test_predictions = test_predictions[pool]

    scores = {
        name: mean_absolute_error(y_test, test_predictions[name])
        for name in pool
    }

    errors = validation_predictions.sub(y_validation, axis=0)
    weights = optimal_weights(errors)
    meta_model = LinearRegression(positive=True).fit(validation_predictions, y_validation)

    scores["Mean"] = mean_absolute_error(y_test, test_predictions.mean(axis=1))
    scores["Median"] = mean_absolute_error(y_test, test_predictions.median(axis=1))
    scores["Optimal weights"] = mean_absolute_error(
        y_test, test_predictions.values @ weights.values
    )
    scores["Stacking"] = mean_absolute_error(y_test, meta_model.predict(test_predictions))

    return scores, weights


EVERYTHING = ["Ridge", "Random forest", "LightGBM", "SARIMAX"]
WITHOUT_FOREST = ["Ridge", "LightGBM", "SARIMAX"]

full, reduced, reduced_weights = [], [], []

for k in range(N_ORIGINS):
    origin = len(X) - HORIZON - (N_ORIGINS - 1 - k) * HORIZON

    # Fit once per origin, then score both pools from the same predictions
    predictions = fit_base_models(X, y, exogenous, origin - VALIDATION_DAYS, origin, HORIZON)
    actual_validation = y.iloc[origin - VALIDATION_DAYS:origin]
    actual_test = y.iloc[origin:origin + HORIZON]

    full.append(score_pool(*predictions, actual_validation, actual_test, EVERYTHING)[0])
    scores, weights = score_pool(*predictions, actual_validation, actual_test, WITHOUT_FOREST)
    reduced.append(scores)
    reduced_weights.append(weights)

labels = [f"origin {k + 1}" for k in range(N_ORIGINS)]
by_origin_full = pd.DataFrame(full, index=labels)
by_origin_reduced = pd.DataFrame(reduced, index=labels)

by_origin_reduced.round(0)

In [ ]:
def summarise(by_origin):
    summary = by_origin.agg(["mean", "std", "max"]).T
    summary.columns = ["Mean MAE", "Std across origins", "Worst origin"]
    return summary.sort_values("Mean MAE")


print("Without the random forest")
print(summarise(by_origin_reduced).round(1).to_string())
print()
print("With it, for comparison")
print(summarise(by_origin_full).round(1).to_string())
print()
print("Best and worst finish of each method across the six origins, forest included:")
print(by_origin_full.rank(axis=1).astype(int).T.assign(
    best=lambda frame: frame.min(axis=1), worst=lambda frame: frame.max(axis=1)
)[["best", "worst"]].to_string())

**No. LightGBM alone averages 369.8 across the six origins and the best combination averages 370.8 — a
tie, with the single model marginally ahead.** Every other combination is worse: stacking 377.6, the median
424.3, the simple mean 437.6.

Then look at the second table, which is the part that makes the exercise worth doing. **With the random
forest in the pool, the answer is the same.** Optimal weights average 358.3 and the random forest alone
averages 358.6. The notebook's headline ensemble was never really beating its best member either; it was
tying it.

So the combination is not failing because the forest left. It is failing for a reason that was already
true.

In [ ]:
print("Weights chosen without the forest, averaged over origins:")
print(pd.DataFrame(reduced_weights).mean().round(3).to_string())
print()
print("Validation error correlation, on the notebook's own split:")
print(validation_errors.corr().round(2).to_string())

The weights say it plainly: with the forest gone, the optimiser puts **0.92 on LightGBM** and hands the
remainder to Ridge and SARIMAX only reluctantly. A weighted average that puts 92% on one model *is* that
model, and it scores accordingly.

That is what determines when combination pays, and it has two conditions that both have to hold:

1. **The models must be comparably good.** Averaging a strong forecaster with a much weaker one moves the
   result towards the weaker one. Here Ridge and SARIMAX are 40% worse than LightGBM across origins, so
   there is no weight above zero that improves on LightGBM alone — which is exactly what the optimiser
   concludes. The simple mean, which has no such discretion, is punished hard for it: 437.6 against 369.8,
   because it gives two bad models a third of the weight each.
2. **They must be wrong at different times.** This is the condition that fails even for the two trees. The
   random forest and LightGBM correlate at **0.93** on validation errors: same features, same splits, same
   mistakes. Averaging two forecasts that miss together reduces almost nothing, which is why the full pool
   also only managed a tie.

**The pool in this notebook fails both conditions** — two near-identical trees and two much weaker models
— so the elaborate weighting machinery has nothing to work with. That is not an argument against
ensembling; it is a statement about what ensembling needs. A pool of a gradient booster, a neural network
from Part D, and a statistical model of comparable skill would satisfy both, and is where the gains
described in the forecasting literature come from.

Two observations that survive the disappointing headline, though, and are worth keeping:

- **Combination is reliably better than choosing badly.** The last table above ranks all eight methods at
  every origin: the weighted ensemble never finishes worse than fourth of eight, while the random forest —
  the best individual model on average — ranges from first to sixth. If you cannot be confident of picking the right model in advance — and origin to origin
  the ranking does move — a combination is a cheap way of not being badly wrong.
- **The learned weights degrade gracefully.** When the best model was removed the optimiser reallocated to
  the next best and lost 12 MAE. The simple mean lost 29, and the median 40. It is precisely when the pool
  is bad that learning the weights earns its keep, which is the opposite of when you would expect it to
  matter.

And the caution from Notebook [A06](../notebooks/A06_Evaluating_models.ipynb) applies in full: the standard
deviation across origins is about 140 MAE, so a 1-point difference between LightGBM and the ensemble means
nothing at all. What the six origins support is "these are equivalent", not "the single model won".

---

Back to [Notebook C03](../notebooks/C03_Ensembles.ipynb), or on to
[Notebook D01](../notebooks/D01_Neural_network_foundations.ipynb).